# Group project · the check cell

The **check cell** is a fixed piece of code that tests your dataset and your baseline results with five tests (T1–T5 in the project brief). Think of it as the technical inspection for a car: it does not say whether your car is beautiful, only whether it is safe to drive. It prints PASS or FAIL per test and, below each line, the numbers it used, so a FAIL tells you where to look.

This notebook shows the check cell working on the UrbanRide bike data. Run it top to bottom once (Runtime → Run all), then copy **the last cell** into your own project notebook, unchanged.

**What the check cell needs from you:** two names, defined before it runs.

| Name | What it is |
|---|---|
| `DATA_DIR` | the folder that holds your ten CSV files |
| `baseline` | your baseline results, as a list of 10 dictionaries (one per seed 1–10) or one dictionary with the means |

## Step 1 · Data: the bike files as a stand-in for yours

This cell copies the course repository into Colab and points `DATA_DIR` at the bike data. The bike folder has no `sources.csv`, so the cell writes a two-row example; yours has one row per number you entered. In your own notebook, `DATA_DIR` is the folder where you uploaded your files.

In [ ]:
import os
if not os.path.exists("sscm-2026-student"):
    !git clone -q https://github.com/TH-Koln-Bartnik/sscm-2026-student
DATA_DIR = "sscm-2026-student/week02/project_bike_data"

# example sources.csv: two rows only, for the demo
with open(os.path.join(DATA_DIR, "sources.csv"), "w") as f:
    f.write("file,row_id,column,value,source_type,source_or_calculation,sanity_check\n"
            "products.csv,UR-CITY,list_price_eur,650,analogy,Example row: city bikes of this class sell for 600-800 EUR,within the range of three web shops\n"
            "routing.csv,WC3,process_time_min,16,guesstimate,Example row: 4 assembly steps x 4 min timed on a video,matches the bike data\n")
print(sorted(os.listdir(DATA_DIR)))

## Step 2 · Baseline results: the eight numbers your model must report

Your model notebook produces these numbers in the baseline scenario. The values below are the means of ten seeds of a simple bike-line model; yours come from your own model.

| Key | Meaning | How to measure it in your model |
|---|---|---|
| `weeks_simulated` | weeks in the run | 52 |
| `hours_simulated` | length of the run in clock hours | e.g. 52 × 5 days × 8 h = 2,080 if your clock counts working hours only |
| `units_started` | units that entered the line | count at the first station |
| `units_finished` | units that left the last station | count at the last station |
| `key_components_used` | key components taken from stock into the line | count where the component enters (`added_at_workcenter`) |
| `avg_units_on_line` | average number of units between start and finish, over the whole run | see the helper below |
| `avg_hours_on_line` | average hours from start to finish of a finished unit | finish time − start time, averaged |
| `cost_per_unit_eur` | one-year total cost ÷ units finished | the total-cost formula of the brief |

**Use one clock for everything.** If `hours_simulated` counts working hours only, `avg_hours_on_line` must count working hours too. Little's Law (T3) compares the two, and a mixed clock is the most common reason for a FAIL.

In [ ]:
baseline = {
    "weeks_simulated": 52,
    "hours_simulated": 2080,
    "units_started": 13866,
    "units_finished": 13853.6,
    "key_components_used": 13866,
    "avg_units_on_line": 12.44,
    "avg_hours_on_line": 1.87,
    "cost_per_unit_eur": 314.58,     # illustration: materials + station fixed costs only
}

### Optional helper: average units on the line from start and finish times

If your model records, for every unit, the hour it started and the hour it finished, this function returns the two averages Little's Law needs. A unit that has not finished by the end of the run counts as "on the line" until the end.

In [ ]:
import numpy as np

def line_averages(start_h, finish_h, hours_simulated):
    """start_h, finish_h: one entry per unit started (use np.inf for unfinished units)."""
    start_h, finish_h = np.asarray(start_h, float), np.asarray(finish_h, float)
    done = finish_h <= hours_simulated
    avg_units_on_line = (np.minimum(finish_h, hours_simulated) - start_h).sum() / hours_simulated
    avg_hours_on_line = (finish_h[done] - start_h[done]).mean()
    return round(float(avg_units_on_line), 2), round(float(avg_hours_on_line), 2)

# tiny example: 3 units on a 10-hour clock, the third one still on the line at the end
print(line_averages([0, 2, 8], [3, 6, np.inf], 10))

## Step 3 · The check cell

Copy the whole next cell into your project notebook, **unchanged**, below your baseline run. The last output line carries two fingerprints: `cell` shows whether the cell was edited, `data` identifies your exact CSV files. Paste the full printed output into your appendix and upload it at the checkpoint.

In [ ]:
# ===================== SSCM 2026 · GROUP PROJECT CHECK CELL · v1.0 =====================
# Copy this cell into your notebook UNCHANGED and run it after your baseline run.
# It needs two names defined earlier in your notebook:
#   DATA_DIR = the folder that holds your CSV files, e.g. DATA_DIR = "my_product_data"
#   baseline = your baseline results: one dictionary per seed (a list of 10), or one
#              dictionary with the means over the seeds. Keys (all numbers):
#     weeks_simulated      weeks in the run (52)
#     hours_simulated      clock hours of the run; the same clock as avg_hours_on_line
#     units_started        units that entered the line
#     units_finished       units that left the last station
#     key_components_used  key components taken from stock into the line
#     avg_units_on_line    time-average of units between start and finish
#     avg_hours_on_line    average hours from start to finish of a finished unit
#     cost_per_unit_eur    one-year total cost divided by units finished
# The grader runs the original of this cell on your files. Do not edit it.
import hashlib, math, os, datetime
import pandas as pd

CHECK_VERSION = "1.0"
REQUIRED = {
    "products.csv": ["product_id", "product_name", "frame_material", "mass_kg", "list_price_eur"],
    "bom.csv": ["product_id", "component_id", "component_name", "qty_per_unit", "mass_kg",
                "unit_cost_eur", "supplier_id", "added_at_workcenter"],
    "workcenters.csv": ["workcenter_id", "workcenter_name", "seq", "num_servers", "shift_hours",
                        "days_per_week", "availability", "process_time_cv", "fixed_cost_eur_week"],
    "routing.csv": ["product_id", "seq", "workcenter_id", "process_time_min", "setup_time_min"],
    "process_environment.csv": ["workcenter_id", "kw_running", "kw_idle", "waste_kg_per_unit",
                                "water_l_per_unit"],
    "suppliers.csv": ["supplier_id", "supplier_name", "country", "component_id", "unit_price_eur",
                      "promised_lead_days", "lead_time_mean_days", "lead_time_sd_days",
                      "capacity_per_month", "min_per_month", "kg_co2e_per_unit", "location_id"],
    "locations.csv": ["location_id", "name", "type", "country", "lat", "lon"],
    "network_lanes.csv": ["lane_id", "origin_id", "destination_id", "mode_id", "distance_km",
                          "transit_days_mean", "transit_days_sd"],
    "freight_rates.csv": ["lane_id", "mode_id", "rate_eur_per_shipment", "rate_eur_per_tkm",
                          "min_charge_eur"],
    "sources.csv": ["file", "row_id", "column", "value", "source_type", "source_or_calculation",
                    "sanity_check"],
}
MAY_BE_BLANK = {("bom.csv", "supplier_id")}           # parts not bought from a modelled supplier
MAY_BE_NEGATIVE = {("locations.csv", "lat"), ("locations.csv", "lon")}
KEYS = ["weeks_simulated", "hours_simulated", "units_started", "units_finished",
        "key_components_used", "avg_units_on_line", "avg_hours_on_line", "cost_per_unit_eur"]

def _cell_fingerprint():
    try:
        return hashlib.sha256(get_ipython().user_ns["_ih"][-1].strip().encode()).hexdigest()[:12]
    except Exception:
        return "n/a"

def _runs():
    b = globals().get("baseline")
    runs = b if isinstance(b, (list, tuple)) else [b]
    if b is None or not runs:
        raise ValueError("no `baseline` defined: run your baseline first")
    for i, r in enumerate(runs, 1):
        missing = [k for k in KEYS if k not in r]
        if missing:
            raise ValueError(f"baseline run {i} lacks {missing}")
        for k in KEYS:
            if not isinstance(r[k], (int, float)) or not math.isfinite(r[k]) or r[k] < 0:
                raise ValueError(f"baseline run {i}: {k} = {r[k]!r} is not a number ≥ 0")
        if r["units_finished"] <= 0 or r["hours_simulated"] <= 0 or r["weeks_simulated"] <= 0:
            raise ValueError(f"baseline run {i}: units_finished, hours and weeks must be > 0")
    return runs

def _verdict(name, ok, lines):
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    for line in lines:
        print("      " + line)
    return ok

data, results = {}, {}
folder = globals().get("DATA_DIR", ".")
h = hashlib.sha256()

# ---- T1 Format ----------------------------------------------------------------------
problems = []
for fname, cols in REQUIRED.items():
    path = os.path.join(folder, fname)
    if not os.path.exists(path):
        problems.append(f"{fname}: file missing in {folder!r}")
        continue
    h.update(open(path, "rb").read())
    df = pd.read_csv(path)
    df.columns = [str(c) for c in df.columns]
    data[fname] = df
    lacking = [c for c in cols if c not in df.columns]
    if lacking:
        problems.append(f"{fname}: column(s) missing or misspelt: {lacking}")
    if len(df) == 0:
        problems.append(f"{fname}: no rows")
    for c in [c for c in cols if c in df.columns]:
        blanks = df[c].isna() | (df[c].astype(str).str.strip() == "")
        if blanks.any() and (fname, c) not in MAY_BE_BLANK:
            problems.append(f"{fname}: {c} is empty in row(s) {list(df.index[blanks] + 2)}")
        num = pd.to_numeric(df[c], errors="coerce")
        if (num < 0).any() and (fname, c) not in MAY_BE_NEGATIVE:
            problems.append(f"{fname}: {c} is negative in row(s) {list(df.index[num < 0] + 2)}")
if "workcenters.csv" in data and "availability" in data["workcenters.csv"]:
    av = pd.to_numeric(data["workcenters.csv"]["availability"], errors="coerce")
    if ((av <= 0) | (av > 1) | av.isna()).any():
        problems.append("workcenters.csv: availability must be a share between 0 and 1")
if "routing.csv" in data and "process_time_min" in data["routing.csv"]:
    if (pd.to_numeric(data["routing.csv"]["process_time_min"], errors="coerce") <= 0).any():
        problems.append("routing.csv: every process_time_min must be above 0")
results["T1"] = _verdict("T1 Format", not problems,
                         problems[:12] + (["… and more"] if len(problems) > 12 else [])
                         or ["all 10 files present, columns exact, no blanks or negatives"])

# ---- baseline numbers for T2–T5 -------------------------------------------------------
try:
    runs = _runs()
    base_error = None
except Exception as e:
    runs, base_error = [], str(e)
if base_error:
    for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
        results[t[:2]] = _verdict(t, False, [base_error])
else:
    try:
        prod = data["products.csv"].iloc[0]
        pid = prod["product_id"]
        wc = data["workcenters.csv"].set_index("workcenter_id")
        rt = data["routing.csv"]
        rt = rt[rt["product_id"] == pid]
        minutes = rt.groupby("workcenter_id")["process_time_min"].sum()
        unknown = [w for w in minutes.index if w not in wc.index]
        avail_min_week = (wc["num_servers"] * wc["shift_hours"] * 60 * wc["days_per_week"]
                          * wc["availability"])
        # ---- T2 Capacity: load offered by the units started, per station
        lines, ok = [], not unknown and len(minutes) > 0
        if unknown:
            lines.append(f"routing.csv names stations not in workcenters.csv: {unknown}")
        worst = {}
        for r in runs:
            for w, m in minutes.items():
                if w in wc.index:
                    share = r["units_started"] * m / (avail_min_week[w] * r["weeks_simulated"])
                    worst[w] = max(worst.get(w, 0), share)
        for w, share in worst.items():
            flag = "  <- 100 % or more" if share >= 1 else ""
            lines.append(f"{w} {wc.loc[w, 'workcenter_name']}: busy {share:.1%} of available time{flag}")
            ok = ok and share < 1
        results["T2"] = _verdict("T2 Capacity", ok, lines)
        # ---- T3 Little's Law: L = λ × W, within 10 %
        lines, ok = [], True
        for i, r in enumerate(runs, 1):
            lam = r["units_finished"] / r["hours_simulated"]
            lw = lam * r["avg_hours_on_line"]
            gap = abs(r["avg_units_on_line"] - lw) / lw if lw > 0 else float("inf")
            ok = ok and gap <= 0.10
            if (gap > 0.10 and len(lines) < 3) or i == 1:
                lines.append(f"run {i}: L = {r['avg_units_on_line']:.2f}  vs  λ×W = {lam:.3f}/h × "
                             f"{r['avg_hours_on_line']:.2f} h = {lw:.2f}  (gap {gap:.1%})")
        results["T3"] = _verdict("T3 Little's Law", ok, lines + [f"{len(runs)} run(s) checked"])
        # ---- T4 Flow balance
        bom = data["bom.csv"]
        bom = bom[bom["product_id"] == pid]
        keys = bom[bom["component_id"].isin(data["suppliers.csv"]["component_id"])]
        lines, ok = [], len(keys) == 1
        if len(keys) != 1:
            lines.append(f"expected one key component (in bom.csv and suppliers.csv), found {len(keys)}")
        else:
            q = float(keys["qty_per_unit"].iloc[0])
            n_opt = int((data["suppliers.csv"]["component_id"] == keys["component_id"].iloc[0]).sum())
            if n_opt < 2:
                lines.append(f"note: only {n_opt} sourcing option in suppliers.csv (the brief asks for two)")
            for i, r in enumerate(runs, 1):
                s, f, u = r["units_started"], r["units_finished"], r["key_components_used"]
                good = f <= s + 0.5 and f * q - 0.5 <= u <= s * q + 0.5
                ok = ok and good
                if not good or i == 1:
                    lines.append(f"run {i}: started {s:,.0f}, finished {f:,.0f}, key components used "
                                 f"{u:,.0f} (allowed {f*q:,.0f} to {s*q:,.0f})")
        results["T4"] = _verdict("T4 Flow balance", ok, lines)
        # ---- T5 Margin
        price = float(prod["list_price_eur"])
        material = float((bom["qty_per_unit"] * bom["unit_cost_eur"]).sum())
        costs = [r["cost_per_unit_eur"] for r in runs]
        lines = [f"cost per unit {min(costs):,.2f} to {max(costs):,.2f} EUR vs list price {price:,.2f} EUR"]
        if min(costs) < material:
            lines.append(f"warning: below the material cost alone ({material:,.2f} EUR per unit)")
        results["T5"] = _verdict("T5 Margin", max(costs) < price, lines)
    except Exception as e:
        for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
            if t[:2] not in results:
                results[t[:2]] = _verdict(t, False, [f"could not run: {e}"])

print("-" * 78)
print(f"{sum(results.values())} of 5 tests passed  ·  {'ALL PASS' if all(results.values()) and len(results) == 5 else 'NOT YET'}")
print(f"check cell v{CHECK_VERSION} · cell {_cell_fingerprint()} · data {h.hexdigest()[:12]} · "
      f"{datetime.datetime.now():%Y-%m-%d %H:%M}")
# ======================= end of check cell · do not edit above ==========================

## What to do with a FAIL

| FAIL in | Typical cause | Where to look |
|---|---|---|
| T1 Format | a column name with a typo, an empty cell, a minus sign, a file not uploaded | the line names the file, the column and the row |
| T2 Capacity | the line starts more units than its slowest station can handle | raise servers, shift hours or availability at that station, or start fewer units |
| T3 Little's Law | two clocks mixed (working hours vs calendar hours), or units counted in some places and not in others | measure both averages over the same units and the same clock |
| T4 Flow balance | units created without a key component, or components consumed twice | count the key component where it enters the line |
| T5 Margin | a cost counted per week instead of per year, or a price too low | recompute the total cost line by line |

_Prepared with Claude (Anthropic) as a coding assistant; checked and released by Professor Bartnik._